# DuckDB Benchmark

This notebook runs an representative end to end use case over data sourced from the [UK Land Registry House Price Data open data repository](https://www.gov.uk/government/statistical-data-sets/price-paid-data-downloads).

This data is made available for us under an [Open Government Licence](https://www.nationalarchives.gov.uk/doc/open-government-licence/version/3/).

We will run two processes:

1. Load raw data, clean it up, add new features and finally write it as a mini dimensional model to lakehouse.
1. Query two of the tables in the dimensional model, join them and summarise the data.

DuckDB is an in-process SQL OLAP database that excels at analytical queries. It provides a familiar SQL interface while delivering excellent performance on single-node workloads.

In [49]:
import duckdb
import time
import logging

In [50]:
logger = logging.getLogger(name="duckdb_benchmark_notebook")
logger.setLevel(logging.INFO)

In [51]:
# Create an in-memory DuckDB connection
# For persistence, use duckdb.connect('my_database.db')
con = duckdb.connect()

In [52]:
# Install and load Delta Lake extension for reading/writing Delta tables
con.execute("INSTALL delta;")
con.execute("LOAD delta;")

In [53]:
from datetime import datetime

source_path = "../../data/fabric/Files/land_registry_data" # ABFSS path to location where raw data (multiple CSV files) is stored

# Add timestamp to paths to avoid overwrite issues with Parquet
run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
target_path_prices = f"../../data/fabric/Tables/duckdb_benchmark/{run_timestamp}/prices.parquet"
target_path_locations = f"../../data/fabric/Tables/duckdb_benchmark/{run_timestamp}/locations.parquet"
target_path_dates = f"../../data/fabric/Tables/duckdb_benchmark/{run_timestamp}/dates.parquet"

In [54]:
start = time.perf_counter()

In [55]:
logging.info(f"Reading price paid data from location {source_path}...")

# DuckDB can read multiple CSV files using glob patterns
# Create a view from the CSV files with explicit column names
con.execute(f"""
    CREATE OR REPLACE VIEW price_paid_raw AS
    SELECT 
        column00 AS transaction_unique_identifier,
        CAST(column01 AS DOUBLE) AS price,
        CAST(column02 AS TIMESTAMP) AS date_of_transfer,
        column03 AS postcode,
        column04 AS property_type,
        column05 AS old_new,
        column06 AS duration,
        column07 AS paon,
        column08 AS saon,
        column09 AS street,
        column10 AS locality,
        column11 AS town_city,
        column12 AS district,
        column13 AS county,
        column14 AS ppd_category_type,
        column15 AS record_status
    FROM read_csv(
        '{source_path}/*.csv',
        header=false,
        nullstr=''
    )
""")

## Data Transformation

With DuckDB, we use SQL to transform our data. Views allow us to build up transformations lazily - the actual computation happens when we query the final result.

In [56]:
# Apply all transformations in a single SQL statement:
# - Convert property_type codes to full descriptions
# - Convert old_new codes to full descriptions  
# - Extract postcode area using regex
# - Convert date_of_transfer to date type

con.execute("""
    CREATE OR REPLACE VIEW price_paid_data AS
    SELECT
        transaction_unique_identifier,
        price,
        CAST(date_of_transfer AS DATE) AS date_of_transfer,
        postcode,
        CASE property_type
            WHEN 'D' THEN 'Detached'
            WHEN 'S' THEN 'Semi-Detached'
            WHEN 'T' THEN 'Terraced'
            WHEN 'F' THEN 'Flat/Maisonette'
            WHEN 'O' THEN 'Other'
            ELSE property_type
        END AS property_type,
        CASE old_new
            WHEN 'Y' THEN 'New'
            WHEN 'N' THEN 'Old'
            ELSE old_new
        END AS old_new,
        duration,
        paon,
        saon,
        street,
        locality,
        town_city,
        district,
        county,
        ppd_category_type,
        record_status,
        regexp_extract(postcode, '^([A-Z]{1,2})', 1) AS postcode_area
    FROM price_paid_raw
""")

### Create fact table

Select the core columns we want to use in the core fact table.

In [57]:
# Create prices view with selected columns
con.execute("""
    CREATE OR REPLACE VIEW prices AS
    SELECT
        price,
        date_of_transfer,
        postcode_area,
        town_city,
        property_type,
        old_new
    FROM price_paid_data
""")

### Create date dimension

Use min and max dates to build date dimension table.

DuckDB's `generate_series` function makes it easy to create date ranges.

In [58]:
# Get min and max dates
date_range = con.execute("""
    SELECT 
        MIN(date_of_transfer) AS min_date,
        MAX(date_of_transfer) AS max_date
    FROM price_paid_data
""").fetchone()

min_date, max_date = date_range
min_date, max_date

(datetime.date(2023, 1, 1), datetime.date(2025, 11, 28))

In [59]:
# Create date dimension table using generate_series
con.execute(f"""
    CREATE OR REPLACE VIEW dates AS
    SELECT
        date::DATE AS date,
        EXTRACT(YEAR FROM date)::INTEGER AS year,
        EXTRACT(MONTH FROM date)::INTEGER AS month,
        strftime(date, '%B') AS month_name,
        EXTRACT(DAY FROM date)::INTEGER AS day,
        EXTRACT(DAYOFWEEK FROM date)::INTEGER AS weekday,
        strftime(date, '%A') AS weekday_name,
        EXTRACT(DAYOFYEAR FROM date)::INTEGER AS day_of_year
    FROM generate_series(
        DATE '{min_date}',
        DATE '{max_date}',
        INTERVAL 1 DAY
    ) AS t(date)
""")

### Create location dimension

Assumption is there is a hierarchy in decreasing order of granularity:

- County
- District
- Town or City

In [60]:
# Create locations view with unique combinations
con.execute("""
    CREATE OR REPLACE VIEW locations AS
    SELECT DISTINCT
        county,
        district,
        town_city
    FROM price_paid_data
""")

## Writing to Delta Tables

DuckDB supports writing to Delta Lake format using the delta extension.

Write modes available:

```sql
-- Overwrite entire table (default behavior with COPY)
COPY table_name TO 'path' (FORMAT DELTA);

-- For more control, use the delta extension functions
```

Note: DuckDB's Delta support is still evolving. For production use on Fabric, you may want to use Parquet format instead, which has mature support.

### Writing with Parquet (Alternative)

For broader compatibility, we can also write to Parquet format:

```sql
COPY prices TO 'path/prices.parquet' (FORMAT PARQUET);
```

### Write tables

In [61]:
import os

# Ensure target directory exists
os.makedirs(os.path.dirname(target_path_prices), exist_ok=True)

In [62]:
logger.info(f"Writing prices data to Parquet: {target_path_prices}")
con.execute(f"COPY prices TO '{target_path_prices}' (FORMAT PARQUET)")

INFO:duckdb_benchmark_notebook:Writing prices data to Parquet: ../../data/fabric/Tables/duckdb_benchmark/20260120_182318/prices.parquet


In [63]:
logger.info(f"Writing locations data to Parquet: {target_path_locations}")
con.execute(f"COPY locations TO '{target_path_locations}' (FORMAT PARQUET)")

INFO:duckdb_benchmark_notebook:Writing locations data to Parquet: ../../data/fabric/Tables/duckdb_benchmark/20260120_182318/locations.parquet


In [64]:
logger.info(f"Writing dates data to Parquet: {target_path_dates}")
con.execute(f"COPY dates TO '{target_path_dates}' (FORMAT PARQUET)")

INFO:duckdb_benchmark_notebook:Writing dates data to Parquet: ../../data/fabric/Tables/duckdb_benchmark/20260120_182318/dates.parquet


## Reading from Delta Lake and generate summary

DuckDB can read Delta tables directly using the `delta_scan` function.

Let's generate some analytics using the data we have just written.

In [65]:
# Load prices from Parquet and filter out "Other" property types
logger.info(f"Reading prices data back from Parquet: {target_path_prices}")
con.execute(f"""
    CREATE OR REPLACE VIEW prices_filtered AS
    SELECT *
    FROM read_parquet('{target_path_prices}')
    WHERE property_type != 'Other'
""")

INFO:duckdb_benchmark_notebook:Reading prices data back from Parquet: ../../data/fabric/Tables/duckdb_benchmark/20260120_182318/prices.parquet


In [66]:
# Load the date dimension with month_tag column
logger.info(f"Reading dates data back from Parquet: {target_path_dates}")
con.execute(f"""
    CREATE OR REPLACE VIEW dates_with_tag AS
    SELECT 
        *,
        strftime(date, '%Y_%m') AS month_tag
    FROM read_parquet('{target_path_dates}')
""")

INFO:duckdb_benchmark_notebook:Reading dates data back from Parquet: ../../data/fabric/Tables/duckdb_benchmark/20260120_182318/dates.parquet


In [67]:
# Join prices with dates and create monthly summary in one query
# This demonstrates DuckDB's ability to compose complex analytical queries
monthly_summary = con.execute("""
    SELECT
        d.month_tag,
        p.property_type,
        COUNT(*) AS number_of_transactions,
        MEDIAN(p.price) AS median_price,
        MIN(p.price) AS min_price,
        MAX(p.price) AS max_price
    FROM prices_filtered p
    LEFT JOIN dates_with_tag d
        ON p.date_of_transfer = d.date
    GROUP BY
        d.month_tag,
        p.property_type
    ORDER BY
        d.month_tag,
        p.property_type
""").fetchdf()

In [68]:
monthly_summary.head(5)

,month_tag,property_type,number_of_transactions,median_price,min_price,max_price
0,2023_01,Detached,13310,430000.0,33000.0,44750000.0
1,2023_01,Flat/Maisonette,12979,239000.0,14000.0,13500000.0
2,2023_01,Semi-Detached,16639,260000.0,950.0,28000000.0
3,2023_01,Terraced,17796,212500.0,12500.0,16150000.0
4,2023_02,Detached,13104,420000.0,39999.0,17500000.0


In [69]:
elapsed = time.perf_counter() - start
logger.info(f"Notebook completed in {elapsed:.2f} seconds.")

INFO:duckdb_benchmark_notebook:Notebook completed in 1.37 seconds.


In [70]:
# Clean up - close the connection
con.close()

## Summary

DuckDB provides a powerful SQL-based approach to analytical data processing. Key advantages include:

- **Familiar SQL syntax**: Leverage existing SQL knowledge for data transformations
- **Zero-copy integration**: Efficiently works with Pandas, Polars, and Arrow data
- **In-process execution**: No separate server needed, runs embedded in your application
- **Excellent performance**: Columnar storage and vectorized execution engine
- **Rich analytical functions**: Native support for window functions, MEDIAN, PERCENTILE, etc.

DuckDB is an excellent choice when:

- Your team prefers SQL over DataFrame APIs
- You need complex analytical queries with joins and aggregations
- You want to prototype queries that will later run on a SQL data warehouse
- You're working with data that fits in memory on a single node